In [1]:
# RUN THIS CELL TO SETUP THE CHALLENGE DATA

import pandas as pd
import sqlite3

conn_challenge = sqlite3.connect(':memory:')

challenge_data = {
    "Visit_ID": [5001, 5001, 5002, 5003],
    "Student_ID": [101, 101, 102, 104],
    "Student_Name": ["Alice", "Alice", "Bob", "David"],
    "Doctor_ID": ["DOC_XYZ", "DOC_XYZ", "DOC_ABC", "DOC_XYZ"],
    "Doctor_Name": ["Dr. Evans", "Dr. Evans", "Dr. Green", "Dr. Evans"],
    "Doctor_Clinic": ["General Medicine", "General Medicine", "Sports Med", "General Medicine"],
    "Prescriptions": ["Amoxicillin, Ibuprofen", "Amoxicillin, Ibuprofen", "Bandages", "Vitamin D"]
}

df_challenge = pd.DataFrame(challenge_data)

df_challenge.to_sql('Patient_Visits_ONF', conn_challenge, index=False, if_exists='replace')

print("--- Challenge Dataset (ONF) ---")

df_challenge

--- Challenge Dataset (ONF) ---


,Visit_ID,Student_ID,Student_Name,Doctor_ID,Doctor_Name,Doctor_Clinic,Prescriptions
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,"Amoxicillin, Ibuprofen"
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,"Amoxicillin, Ibuprofen"
2,5002,102,Bob,DOC_ABC,Dr. Green,Sports Med,Bandages
3,5003,104,David,DOC_XYZ,Dr. Evans,General Medicine,Vitamin D


TASK 1: 

1. Why does this table violate 1NF? Which column is the culprit?

-- It violates 1NF because values are not atomic (multi-valued fields exist).
-- "Prescriptions" column, it contains multiple values like "Amoxicillin, Ibuprofen"


2. Why do Student_Name and Doctor_Clinic violate 2NF and 3NF?

-- Assuming Visit_ID is the primary key:
-- Student_Name → depends on Student_ID (partial dependency → 2NF violation)
-- Doctor_Clinic → depends on Doctor_ID (transitive dependency → 3NF violation)


3. Which normal form is violated?

-- It violates 3NF because Doctor_Clinic depends on Doctor_ID, which is a non-key attribute, creating a transitive dependency.

In [2]:
# TASK 2: CONVERT TO 1NF
import pandas as pd
import sqlite3

# Create database connection
conn = sqlite3.connect(':memory:')

# Original ONF dataset
challenge_data = {
    "Visit_ID": [5001, 5001, 5002, 5003],
    "Student_ID": [101, 101, 102, 104],
    "Student_Name": ["Alice", "Alice", "Bob", "David"],
    "Doctor_ID": ["DOC_XYZ", "DOC_XYZ", "DOC_ABC", "DOC_XYZ"],
    "Doctor_Name": ["Dr. Evans", "Dr. Evans", "Dr. Green", "Dr. Evans"],
    "Doctor_Clinic": ["General Medicine", "General Medicine", "Sports Med", "General Medicine"],
    "Prescriptions": ["Amoxicillin, Ibuprofen", "Amoxicillin, Ibuprofen", "Bandages", "Vitamin D"]
}

# Convert to DataFrame
df_onf = pd.DataFrame(challenge_data)

#   1. SPLIT MULTI-VALUED COLUMN (1NF RULE)

df_1nf = df_onf.assign(
    Prescriptions=df_onf["Prescriptions"].str.split(", ")
).explode("Prescriptions")

# 2. STORE IN SQL TABLE

df_1nf.to_sql('Patient_Visits_1NF', conn, index=False, if_exists='replace')

print("--- 1NF Normalized Table ---")
df_1nf

--- 1NF Normalized Table ---


,Visit_ID,Student_ID,Student_Name,Doctor_ID,Doctor_Name,Doctor_Clinic,Prescriptions
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Amoxicillin
0,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Ibuprofen
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Amoxicillin
1,5001,101,Alice,DOC_XYZ,Dr. Evans,General Medicine,Ibuprofen
2,5002,102,Bob,DOC_ABC,Dr. Green,Sports Med,Bandages
3,5003,104,David,DOC_XYZ,Dr. Evans,General Medicine,Vitamin D
